# Model Results

Fits and evalutes the Bayesian model.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, RANDOM_SEED, TABLES_DIR, TEST_SEASONS
from src.data.add_team_id import build_team_to_id_mapping, add_team_ids
from src.model.model import BayesianPoissonModel, BayesianPoissonModelConfig, SVIConfig

/Users/astonchan/Desktop/Academics Folder/UC Irvine/Senior Year/Spring Quarter/COMPSCI 179 – Algorithms for Probabilistic and Deterministic Graphical Models/Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation

In [4]:
TRAIN_SEASONS = ["20-21", "21-22", "22-23", "23-24"]

### Load Training and Testing Data

In [5]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates=["Date"])

df = df.drop(columns = ["HomeTeamID", "AwayTeamID"], errors = "ignore")
team_mapping = build_team_to_id_mapping(df, TRAIN_SEASONS)
df = add_team_ids(df, team_mapping)

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop=True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop=True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 1520 matches
Test: 760 matches


### Bayesian Model

In [ ]:
bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = False,
        ablate_defense = False,
        ablate_home_team_advantage = False,
        num_posterior_samples_per_match = 100,
    )
).fit(train_df)
bayesian_model_preds = bayesian_model.predict(test_df)

Step 100: Loss = 4792.672
Step 200: Loss = 4738.481
Step 300: Loss = 4734.071
Step 400: Loss = 4728.928
Step 500: Loss = 4731.848
Step 600: Loss = 4729.912
Step 700: Loss = 4725.293
Step 800: Loss = 4723.405
Step 900: Loss = 4726.572
Step 1000: Loss = 4721.698
Step 1100: Loss = 4724.320
Step 1200: Loss = 4724.815
Step 1300: Loss = 4726.053
Step 1400: Loss = 4724.544
Step 1500: Loss = 4724.447
Step 1600: Loss = 4724.531
Step 1700: Loss = 4727.886
Step 1800: Loss = 4725.231
Step 1900: Loss = 4725.083


In [ ]:
bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

In [ ]:
pred_labels = bayesian_model_preds[["ProbHomeWin",
                                    "ProbDraw",
                                    "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                          "ProbDraw": "D",
                                                                          "ProbAwayWin": "A"})
accuracy = (pred_labels == bayesian_model_preds["Result"]).mean()
print(f"Bayesian Model Accuracy: {accuracy:.3f}")

### Save Evaluation Results

In [ ]:
model_table_output_directory = (TABLES_DIR / "model")
model_table_output_directory.mkdir(parents = True, exist_ok = True)

bayesian_model_preds.to_csv(model_table_output_directory / "training_window_bayesian_model_predictions.csv",
                            index = False)